In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')

from Model.Constants import db, model_db
from Model.DBTypes import *
from Model.ModelDBTypes import *
from Model.DB_AdvancedStatements import *

In [ ]:
cursor = db.cursor()

hitter_set = {0}
hitter_first_months : list[tuple[DB_Model_HitterStats, DB_HitterStatcastMonth]] = []

years = [2021, 2022, 2023]
months = [4,5,6,7,8,9]

for year in years:
    for month in months:
        low_a_hitters = Select_InnerJoin(DB_Model_HitterStats, DB_HitterStatcastMonth, cursor,
            '''SELECT mhs.*, psm.*
            FROM Model_HitterStats AS mhs
            INNER JOIN HitterStatcastMonth AS psm
                ON mhs.mlbId=psm.mlbId
                AND mhs.Year=psm.Year
                AND mhs.Month=psm.Month
            WHERE mhs.Year=?
            AND mhs.Month=?
            AND mhs.LevelId>=5
            AND mhs.PA>30
            ''',
            (year, month)
        )
        
        for h in low_a_hitters:
            mhs, hsm = h
            if not mhs.mlbId in hitter_set:
                hitter_set.add(mhs.mlbId)
                hitter_first_months.append(h)

print(len(hitter_first_months))

In [ ]:
model_cursor = model_db.cursor()

data : list[tuple[float, float]] = []

for mhs, hsm in hitter_first_months:
    try:
        expected_war = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
            "WHERE mlbId=? AND model=1 AND isHitter=1 AND Year=? AND Month=?",
            (mhs.mlbId, mhs.Year, mhs.Month))[0].war
        
        try:
            expected_war_2years = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
                "WHERE mlbId=? AND model=1 AND isHitter=1 AND Year=? AND Month=?",
                (mhs.mlbId, mhs.Year + 2, mhs.Month))[0].war
        except:
            # Do last value
            expected_war_2years = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
                "WHERE mlbId=? AND model=1 AND isHitter=1 ORDER BY Year DESC, Month DESC",
                (mhs.mlbId,))[0].war
            
        val = hsm.ChangeupContactPerc
        if expected_war < 3 and val is not None:
            data.append((val, expected_war_2years - expected_war))
    except:
        pass # Not a prospect
    


In [ ]:
def sortFunction(d : tuple[float, float]):
    x, y = d
    return x

data.sort(key=sortFunction)

In [ ]:
l = len(data)
NUM_BINS = 10

quantile_cutoffs = [data[int(l * x / NUM_BINS)][0] for x in range(1, NUM_BINS)]

In [ ]:
bins = [[] for _ in range(NUM_BINS)]

def avg(xs : list[float]) -> float:
    if len(xs) == 0:
        return None
    
    return sum(xs) / len(xs)

for val, delta in data:
    if val > quantile_cutoffs[-1]:
        bins[-1].append(delta)
    else:
        for i, quant_val in enumerate(quantile_cutoffs):
            if val < quant_val:
                bins[i].append(delta)
                break
        
bin_ys = [avg(bin) for bin in bins]
for bin in bin_ys:
    print(f"{bin:.3f}")

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

xs = [d[0] for d in data]
ys = [d[1] for d in data]

x = np.array(xs).reshape(-1, 1)
y = np.array(ys).reshape(-1, 1)

lr = LinearRegression()
lr.fit(x, y)
print(f"Slope (coefficient): {lr.coef_[0][0]:.4f}")
print(f"Intercept: {lr.intercept_[0]:.4f}")
print(f"R² Score: {lr.score(x, y):.4f}")

In [ ]:
min_x = data[0][0]
max_x = data[-1][0]

In [ ]:
neutral_xs = [min_x, max_x]
neutral_ys = [0, 0]

In [ ]:
bin_xs = [(min_x + quantile_cutoffs[0]) / 2] + [(quantile_cutoffs[i] + quantile_cutoffs[i+1]) / 2 for i in range(len(quantile_cutoffs) - 1)] + [(max_x + quantile_cutoffs[-1]) / 2]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(xs, ys, 'ko')
plt.plot(bin_xs, bin_ys, 'r-')
plt.plot(neutral_xs, neutral_ys, 'g--')